In [ ]:
#! pip install llama-index llama-index-vector-stores-chroma llama-index-utils-workflow llama-index-llms-huggingface-api pyvis -U -q


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step

class MyWorkflow(Workflow):
    @step
    async def my_step(step, ev: StartEvent) -> StopEvent:
        return StopEvent(result= "Hello World!")

w= MyWorkflow(timeout=10, verbose=False)
result= await w.run()
result

'Hello World!'

Connet multiple steps

In [4]:
from llama_index.core.workflow import Event

class ProcessingEvent(Event):
    intermediate_result: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> ProcessingEvent:
        # Process initial data
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)

w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()
result

'Finished processing: Step 1 complete'

Loops and Branches (use | op-erator tp specify that the step can return multiple types)

In [11]:
from llama_index.core.workflow import Event
import random


class ProcessingEvent(Event):
    intermediate_result: str


class LoopEvent(Event):
    loop_output: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return ProcessingEvent(intermediate_result="First step complete.")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(verbose=False)
result = await w.run()
result

Bad thing happened
Bad thing happened
Good thing happened


'Finished processing: First step complete.'

Draw all workflows

In [12]:
from llama_index.utils.workflow import draw_all_possible_flows

draw_all_possible_flows(w)

workflow_all_flows.html


### State management

In [14]:
from llama_index.core.workflow import Event, Context
from llama_index.core.agent.workflow import ReActAgent


class ProcessingEvent(Event):
    intermediate_result: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent, ctx: Context) -> ProcessingEvent:
        # Process initial data
        await ctx.store.set("query", "What is the capital city of Nepal?")
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent, ctx: Context) -> StopEvent:
        # Use the intermediate result
        query = await ctx.store.get("query")
        print(f"Query: {query}")
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()
result

Query: What is the capital city of Nepal?


'Finished processing: Step 1 complete'

MultiAgent Workflows

llm = HuggingFaceInferenceAPI(\
    model_name="Qwen/Qwen2.5-Coder-32B-Instruct"\
)\
This fails due to particular LlamaIndex integration/client path.

In [19]:
# from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
# from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI


# def add(a: int, b: int) -> int:
#     """Add two numbers."""
#     return a + b


# def multiply(a: int, b: int) -> int:
#     """Multiply two numbers."""
#     return a * b


# llm = HuggingFaceInferenceAPI(
#     model_name="Qwen/Qwen2.5-Coder-32B-Instruct"
# )


# multiply_agent = ReActAgent(
#     name="multiply_agent",
#     description="An agent that can multiply two integers.",
#     system_prompt="""
# You are a multiplication agent.

# You can multiply two integers using the multiply tool.

# If the user asks you to perform addition, hand off the task
# to the add_agent.
# """,
#     tools=[multiply],
#     llm=llm,
# )


# addition_agent = ReActAgent(
#     name="add_agent",
#     description="An agent that can add two integers.",
#     system_prompt="""
# You are an addition agent.

# You can add two integers using the add tool.

# Use the add tool whenever the user asks for addition.
# """,
#     tools=[add],
#     llm=llm,
# )


# workflow = AgentWorkflow(
#     agents=[multiply_agent, addition_agent],
#     root_agent="multiply_agent",
# )


# response = await workflow.run(
#     user_msg="Can you add 5 and 3?"
# )

# print(response)

Use OpenAILike

In [21]:
import os
HF_TOKEN = os.environ["HF_TOKEN"]

In [22]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.llms.openai_like import OpenAILike


def add(a: int, b: int) -> int:
    return a + b


def multiply(a: int, b: int) -> int:
    return a * b


llm = OpenAILike(
    model="Qwen/Qwen2.5-Coder-32B-Instruct",
    api_base="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
    is_chat_model=True,
    temperature=0.1,
    max_tokens=512, 
)


multiply_agent = ReActAgent(
    name="multiply_agent",
    description="An agent that can multiply two integers.",
    system_prompt="""
You are a multiplication agent.
You can multiply two integers using the multiply tool.
If the user asks you to perform addition, hand off the task to the add_agent.
""",
    tools=[multiply],
    llm=llm,
)


addition_agent = ReActAgent(
    name="add_agent",
    description="An agent that can add two integers.",
    system_prompt="""
You are an addition agent.
You can add two integers using the add tool. Use the add tool whenever the user asks for addition.
""",
    tools=[add],
    llm=llm,
)


workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
)


response = await workflow.run(
    user_msg="Can you add 5 and 3?"
)

print(response)

8
